# Colab Ollama Runtime - Qwen 3 Model Hosting

Hosts Ollama with `qwen3:8b` on Colab's free T4 GPU so your local machine can
use a stronger model. Your code stays on your laptop - no repo is needed here.

## Workflow

1. Run all cells in this notebook (first run downloads Ollama + the model, so it takes a few minutes).
2. The last output prints `OLLAMA_BASE_URL` and `OLLAMA_API_KEY`.
3. On your laptop:
   - switch to Colab: `python switch_llm.py colab <OLLAMA_BASE_URL> <OLLAMA_API_KEY>`
   - run as usual, e.g. `python run_sample.py dbms_analytics_test`
   - switch back: `python switch_llm.py local`
   - check the current backend: `python switch_llm.py status`

> The tunnel URL is public; the `OLLAMA_API_KEY` protects it. Keep it secret.


In [ ]:
import os
import re
import secrets
import subprocess
import time

API_KEY = secrets.token_hex(16)
print(f"OLLAMA_API_KEY={API_KEY}")


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
print("starting ollama serve with API key")
env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_API_KEY": API_KEY}
subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        subprocess.run(
            ["curl", "-s", "-f", "http://localhost:11434/api/tags"],
            capture_output=True,
            timeout=5,
        )
        break
    except Exception:
        time.sleep(2)
print("ollama up")


In [ ]:
!ollama pull qwen3:8b


In [ ]:
!curl -L -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared


In [ ]:
log = open("/tmp/cf.log", "w")
subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=log,
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    text = open("/tmp/cf.log").read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    time.sleep(2)
assert url, "tunnel URL not found in /tmp/cf.log"
print(f"OLLAMA_BASE_URL={url}")
print(f"OLLAMA_API_KEY={API_KEY}")


## On your laptop

Run this (replace the URL and key with the printed values):

```
python switch_llm.py colab https://<id>.trycloudflare.com <api-key>
python run_sample.py dbms_analytics_test
```

Switch back to local Ollama anytime:

```
python switch_llm.py local
```
